# Chapter 4: The Analytics Layer

This notebook sets up the Snowflake warehouse as the analytics layer for historical shipment data. While Snowflake Postgres in Chapter 3 stores live operational records, the Snowflake warehouse accumulates those records over time for trend analysis, delay
reporting and SLA breach tracking.

The analytics Streamlit app in Chapter 7 reads exclusively from the warehouse. The incremental sync pattern introduced here -- pulling new rows from Snowflake Postgres and appending them to the warehouse -- is also used by the analytics app on every dashboard refresh.

## 1. Install Dependencies

All versions are pinned for reproducibility. Run this cell once.

In [1]:
%pip install pandas==3.0.5 \
             psycopg2-binary==2.9.12 \
             snowflake-connector-python==3.15.0 --quiet

print("Install complete.")

Note: you may need to restart the kernel to use updated packages.
Install complete.


## 2. Imports

In [2]:
import os
import psycopg2
import snowflake.connector

from datetime import datetime, timezone

## 3. Configuration

Connection details are read from environment variables. No credentials are hard-coded in the notebook.

In [3]:
SNOWFLAKE_ACCOUNT   = os.environ["SNOWFLAKE_ACCOUNT"]
SNOWFLAKE_USER      = os.environ["SNOWFLAKE_USER"]
SNOWFLAKE_PASSWORD  = os.environ["SNOWFLAKE_PASSWORD"]
SNOWFLAKE_DATABASE  = os.environ["SNOWFLAKE_DATABASE"]
SNOWFLAKE_SCHEMA    = os.environ["SNOWFLAKE_SCHEMA"]
SNOWFLAKE_WAREHOUSE = os.environ["SNOWFLAKE_WAREHOUSE"]

SNOWFLAKE_PG_HOST     = os.environ["SNOWFLAKE_PG_HOST"]
SNOWFLAKE_PG_USER     = os.environ["SNOWFLAKE_PG_USER"]
SNOWFLAKE_PG_PASSWORD = os.environ["SNOWFLAKE_PG_PASSWORD"]
SNOWFLAKE_PG_DBNAME   = os.environ["SNOWFLAKE_PG_DBNAME"]

print("Configuration set.")

Configuration set.


## 4. Connect to Snowflake

We connect using `snowflake.connector` with account, user and password. The account identifier is the part of your Snowflake URL before `.snowflakecomputing.com` -- for example if your URL is `abc123.eu-west-1.snowflakecomputing.com` then your account is `abc123.eu-west-1`.

In [4]:
sf = snowflake.connector.connect(
    account  = SNOWFLAKE_ACCOUNT,
    user     = SNOWFLAKE_USER,
    password = SNOWFLAKE_PASSWORD,
)

sf_cursor = sf.cursor()
sf_cursor.execute("SELECT CURRENT_VERSION()")
print("Connected to Snowflake:", sf_cursor.fetchone()[0])

Connected to Snowflake: 10.31.103


## 5. Create Database and Schema

We create the database and schema that will hold the historical shipment data. `IF NOT EXISTS` makes this cell safe to re-run.

In [5]:
sf_cursor.execute(f"CREATE DATABASE IF NOT EXISTS {SNOWFLAKE_DATABASE}")
sf_cursor.execute(f"USE DATABASE {SNOWFLAKE_DATABASE}")
sf_cursor.execute(f"CREATE SCHEMA IF NOT EXISTS {SNOWFLAKE_SCHEMA}")
sf_cursor.execute(f"USE SCHEMA {SNOWFLAKE_SCHEMA}")

print(f"Database : {SNOWFLAKE_DATABASE}")
print(f"Schema   : {SNOWFLAKE_SCHEMA}")

Database : SUPPLY_CHAIN
Schema   : PUBLIC


## 6. Create Warehouse

The Snowflake warehouse is the compute layer that runs queries. We use an X-SMALL warehouse with auto-suspend and auto-resume -- it spins up when a query arrives and suspends after 60 seconds of inactivity to keep costs low.

In [6]:
sf_cursor.execute(f"""
    CREATE WAREHOUSE IF NOT EXISTS {SNOWFLAKE_WAREHOUSE}
    WITH WAREHOUSE_SIZE      = 'X-SMALL'
         AUTO_SUSPEND        = 60
         AUTO_RESUME         = TRUE
         INITIALLY_SUSPENDED = TRUE
""")

sf_cursor.execute(f"USE WAREHOUSE {SNOWFLAKE_WAREHOUSE}")

print(f"Warehouse: {SNOWFLAKE_WAREHOUSE}")

Warehouse: SUPPLY_CHAIN_WH


## 7. Create the Shipment History Table

The `shipment_history` table mirrors the `shipments` table in Snowflake Postgres but is designed for analytics rather than transactions. It has no primary key constraint since it's append-only, and adds a `synced_at` column recording when each row was pulled from Snowflake Postgres.

We drop and recreate the table on each run so the notebook is safe to re-run from the start during development.

In [7]:
sf_cursor.execute("DROP TABLE IF EXISTS shipment_history")

sf_cursor.execute("""
    CREATE TABLE shipment_history (
        shipment_id      TEXT,
        supplier_id      TEXT,
        warehouse_id     TEXT,
        dist_center_id   TEXT,
        retailer_id      TEXT,
        status           TEXT,
        delay_minutes    INTEGER,
        rerouted         BOOLEAN,
        alt_warehouse_id TEXT,
        recorded_at      TIMESTAMP_TZ,
        synced_at        TIMESTAMP_TZ
    )
""")

print("Table shipment_history created.")

Table shipment_history created.


## 8. Incremental Sync from Snowflake Postgres

The sync function reads rows from Snowflake Postgres that are newer than the latest `recorded_at` already in the warehouse, then appends them to `shipment_history`.

This is the same pattern used by the analytics Streamlit app in Chapter 7 -- the app calls this function on every dashboard refresh to keep the warehouse current. Running it here confirms the pattern works end to end before we add the Kafka layer.

The first run will find no rows since the Kafka consumer hasn't started yet. That's expected -- we're verifying the sync mechanism, not the data.

In [8]:
def get_pg_conn():
    """Return a psycopg2 connection to Snowflake Postgres."""
    conn = psycopg2.connect(
        host     = SNOWFLAKE_PG_HOST,
        user     = SNOWFLAKE_PG_USER,
        password = SNOWFLAKE_PG_PASSWORD,
        dbname   = SNOWFLAKE_PG_DBNAME,
        sslmode  = "require",
        port     = 5432
    )
    conn.autocommit = True
    return conn

def get_last_synced_at(sf_cursor):
    """Return the latest recorded_at already in the warehouse, or epoch if empty."""
    sf_cursor.execute("SELECT MAX(recorded_at) FROM shipment_history")
    result = sf_cursor.fetchone()[0]
    return result if result else datetime(1970, 1, 1, tzinfo=timezone.utc)

def sync_to_warehouse(sf_cursor):
    """Pull new rows from Snowflake Postgres and append to shipment_history."""
    last_synced_at = get_last_synced_at(sf_cursor)
    synced_at      = datetime.now(timezone.utc)

    pg_conn   = get_pg_conn()
    pg_cursor = pg_conn.cursor()

    pg_cursor.execute("""
        SELECT
            shipment_id, supplier_id, warehouse_id, dist_center_id,
            retailer_id, status, delay_minutes, rerouted,
            alt_warehouse_id, recorded_at
        FROM shipments
        WHERE recorded_at > %s
        ORDER BY recorded_at
    """, (last_synced_at,))

    rows = pg_cursor.fetchall()
    pg_cursor.close()
    pg_conn.close()

    if not rows:
        return 0

    BATCH_SIZE = 500
    total = 0
    for i in range(0, len(rows), BATCH_SIZE):
        batch = rows[i:i + BATCH_SIZE]
        values = ", ".join(
            sf_cursor.mogrify(
                "(%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)",
                (*row, synced_at)
            )
            for row in batch
        )
        sf_cursor.execute(f"INSERT INTO shipment_history VALUES {values}")
        total += len(batch)

    return total

rows_synced = sync_to_warehouse(sf_cursor)
print(f"Rows synced to warehouse: {rows_synced}")
print("(Zero is expected here -- the Kafka consumer hasn't started yet.)")

Rows synced to warehouse: 0
(Zero is expected here -- the Kafka consumer hasn't started yet.)


## 9. Verify

We confirm the table exists and check the column definitions match what we expect.

In [9]:
sf_cursor.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_name = 'SHIPMENT_HISTORY'
    ORDER BY ordinal_position
""")

print("Table: shipment_history")
print(f"  {'Column':<20} {'Type'}")
print(f"  {'-'*40}")
for col, dtype in sf_cursor.fetchall():
    print(f"  {col:<20} {dtype}")

sf_cursor.execute("SELECT COUNT(*) FROM shipment_history")
print(f"\nRow count: {sf_cursor.fetchone()[0]}")

Table: shipment_history
  Column               Type
  ----------------------------------------
  SHIPMENT_ID          TEXT
  SUPPLIER_ID          TEXT
  WAREHOUSE_ID         TEXT
  DIST_CENTER_ID       TEXT
  RETAILER_ID          TEXT
  STATUS               TEXT
  DELAY_MINUTES        NUMBER
  REROUTED             BOOLEAN
  ALT_WAREHOUSE_ID     TEXT
  RECORDED_AT          TIMESTAMP_TZ
  SYNCED_AT            TIMESTAMP_TZ

Row count: 0


## 10. Teardown

Close the Snowflake connection. The database, schema, warehouse and table all stay in place -- the analytics Streamlit app in Chapter 7 connects to the same warehouse and calls `sync_to_warehouse()` on every dashboard refresh.

In [10]:
sf_cursor.close()
sf.close()
print("Connection closed.")

Connection closed.
